In [1]:
pip install langchain langchain-google-genai faiss-cpu pypdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load PDF
loader = PyPDFLoader("/Users/kamal/Desktop/AI-Projects/REFRAG/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf")
pages = loader.load()

# Extract text
full_text = " ".join([page.page_content for page in pages])

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(full_text)

print(f"Total chunks: {len(chunks)}")


Total chunks: 6781


In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [5]:
chunk_embeddings = embeddings.embed_documents(chunks)

In [6]:
import faiss
import numpy as np

dim = len(chunk_embeddings[0])
index = faiss.IndexFlatL2(dim)
index.add(np.array(chunk_embeddings).astype("float32"))

def retrieve_chunks(query, top_k=5):
    query_vec = embeddings.embed_query(query)
    distances, indices = index.search(np.array([query_vec], dtype="float32"), top_k)
    retrieved = [chunks[i] for i in indices[0]]
    return retrieved, distances[0]


In [ ]:
def selective_expand(query, top_k=5, threshold=0.5):
    retrieved_chunks, distances = retrieve_chunks(query, top_k)
    expanded = []
    compressed = []

    for chunk, dist in zip(retrieved_chunks, distances):
        sim = 1 / (1 + dist)  
        if sim > threshold:
            expanded.append(chunk)  
        else:
            compressed.append(chunk[:200] + "...")  

    return expanded, compressed


In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import HumanMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

In [17]:
query = "Summarize the key points about Breast Cancer."

expanded, compressed = selective_expand(query)

context = "\n\n".join(expanded + compressed)

messages = [
    HumanMessage(content=f"Answer the following using the context below:\n{context}\n\nQuestion: {query}")
]

In [18]:
response = llm.invoke(messages)
print(response.content)

Based on the provided text, here are the key points about breast cancer:

*   **Variability:** There is no "normal" or "typical" female breast. Breasts vary greatly in shape, size, and texture.
*   **Changes:** Breast tissues change due to hormones, aging, nursing, weight fluctuations, and injury. Different tissue types can respond differently to these changes.
*   **Silent Symptoms:** Many cancers, including breast cancer, may not have early symptoms.
*   **Importance of Screening:** Routine screening tests like breast self-exams and mammograms are crucial for early detection.
*   **Common Concern:** Breast cancer is a primary concern for women experiencing breast lumps or abnormal symptoms.
*   **Diagnosis:**
    *   Any newly discovered breast lump should be checked by a doctor.
    *   Mammograms (X-rays of the breast) and ultrasounds (using sound waves) are used for diagnosis.
    *   The size, shape, and edges of masses can help determine if cancer is present, though dense breast